In [1]:
import os
import sys
import pandas as pd
import numpy as np
import random

from bmtk.builder.networks import NetworkBuilder

from matplotlib import pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline

In [2]:
from neuron.units import ms, mV

In [3]:
# Output file names
directory_name = 'network/recurrent_network/'

# Create direcoty if it doesn't exist
if not os.path.exists(directory_name):
    os.makedirs(directory_name)

In [4]:
### Number of cell models desired
N_IN = 1
N_TC = 1

In [5]:
### Define all the cell models in a dictionary (note dictionaries within a dictionary)
biophysical_models = {
    
    # Cell type. Here we are using a pyrmaidal models from an Scnn1a Cre-line. Other
    # models exist in the components directory if you would like to explore them.
    'TC': { 
        # Number of nodes (neurons) we would like to create
        'N': N_TC,               
        # Key to indicate if the neuron is excitatory or inhibitory (optional)
        'ei': 'e', 
        # Key to store the population name of the model. One can have many
        # models from the same population (here cre-line)
        'pop_name': 'TC', 
        # Attribute to indicate this model is biophysical (instead of LIF for example)
        'model_type': 'biophysical',
        # Attribute to indicate that we will be using a NEURON template file created specially
        # for processing Cell Type Databases (ctdb) neuron models. For different type of models
        # and simulators we will want to change this value (or use our own templates)
        'model_template': 'ctdb:Biophys1.hoc',
        # For special post-processing imperatives and functions can be set in model_processing.
        # In this case we are telling the simulator to specially cut the axon as required for 
        # Cell Types models (built in function). We can also pass in custom functions
        'model_processing': 'aibs_perisomatic',       
        # The morpholgy file for this model (NEURON template). This is stored in the components
        # directory if you would like to explore it more (plus other models). 
        'morphology_file': 'TC_jy170517_A_idA.asc',
        # The dynamics parameters file for this model. Again, this is stored
        # in the components directory with other models if you wish to check them out.
        'dynamics_params': 'cAD_ltb.json',
        # Fixed for every model but needs to be defined. This is the angle of the 
        # morphology reconstruction relative to the pia.
        'rotation_angle_zaxis': 0
    },
    
    # Below are the same attributes but with different values for the PV cell we will be
    # modeling in our network.
    'IN': {
        'N': N_IN,
        'ei': 'i', 
        'pop_name': 'IN',
        'model_type': 'biophysical',
        'model_template': 'ctdb:Biophys1.hoc',
        'model_processing': 'aibs_perisomatic',
        'dynamics_params': 'cAD_ltb.json',
        'morphology_file': 'IN_jy171222_A_idA.ASC',
        'rotation_angle_zaxis': 0
    }
}

In [6]:
def generate_random_positions(N):
    '''
    Generate N random positions.
    N: number of positions to generate
    '''

    x = np.random.random(N)     # x-axis location
    y = np.random.random(N)     # y-axis location
    z = np.random.random(N)     # z-axis location

    positions = np.column_stack((x, y, z))

    return positions

In [7]:

### Create nodes set for each biophysical population

# A network object is created. This will store all the 
# information about the nodes and connections between nodes. The object has
# multiple methods (functions) that we will be using below to build the network
# nodes first and connections second.
net = NetworkBuilder('Interneuron_microcircuit')

# We will loop through every model and create nodes. Note the main function
# below is net.add_nodes()
for model in biophysical_models:
    # Read the first biophysical model parameters dictionary
    params = biophysical_models[model]
    
    n_cells = params.pop('N')
    # Calculate positions for every cell (x, y, z). The called function was created
    # above. It doesn't do anything special other than create positions for nodes
    # from a random number generator. But one can write more sophisticated algorithms
    # for positions if they choose.
    positions = generate_random_positions(n_cells)

    # Adds node populations giving it all the input paramters one would like.
    net.add_nodes(N=n_cells, # Specify the numer of cells belonging to this set of nodes 
                  # for the positions and y rotation we are passing in arrays of size N, which mean
                  # such properties are unique for every cell
                  x=positions[:,0], y=positions[:, 1], z=positions[:, 2],
                  rotation_angle_yaxis=np.random.uniform(0.0, 2*np.pi, n_cells),
                  # The other parameters are shared by all cells of this set in the dictionary
                  **params) # python shortcut for unrolling a dictionary

In [8]:

net.save_nodes(nodes_file_name='nodes.h5', node_types_file_name='node_types.csv', output_dir=directory_name)

In [9]:
# Show the nodes file
from bmtk.analyzer import nodes_table
nodes_table('network/recurrent_network/nodes.h5', 'Interneuron_microcircuit')

C:\Users\yijan\anaconda3\lib\site-packages\bmtk-0.0.8-py3.7.egg\bmtk\simulator\utils\config.py:4: UserWarning: Please use bmtk.simulator.core.simulation_config instead.
  warnings.warn('Please use bmtk.simulator.core.simulation_config instead.')


,node_id,node_type_id,rotation_angle_yaxis,x,y,z
0,0,100,0.688479,0.146809,0.149040,0.200376
1,1,101,4.951121,0.253357,0.584275,0.866910


In [10]:

# Show the node_types file. Note the common column is node_type_id
node_types_DF = pd.read_csv('network/recurrent_network/node_types.csv', sep = ' ')
node_types_DF

,node_type_id,rotation_angle_zaxis,ei,pop_name,dynamics_params,model_template,morphology_file,model_type,model_processing
0,100,0,e,TC,cAD_ltb.json,ctdb:Biophys1.hoc,TC_jy170517_A_idA.asc,biophysical,aibs_perisomatic
1,101,0,i,IN,cAD_ltb.json,ctdb:Biophys1.hoc,IN_jy171222_A_idA.ASC,biophysical,aibs_perisomatic


In [11]:
def distance_connection_handler(source, target, d_max, nsyn_min, nsyn_max):
    '''
    Connect cells that are less than d_max apart
    with a random number of synapses in the 
    interval [nsyn_min, nsyn_max)
    '''
    
    sid = source['node_id']    # Get source id
    tid = target['node_id']    # Get target id
    
    # Avoid self-connections.
    if (sid == tid):
        return None

    # first calculate euclidean distance between cells
    src_positions = np.array([source['x'], source['y'], source['z']])
    trg_positions = np.array([target['x'], target['y'], target['z']])
    separation = np.sqrt(np.sum(src_positions - trg_positions)**2 )

    # drop the connection if nodes too far apart
    if separation >= d_max:
        return

    # Add the number of synapses for every connection.
    tmp_nsyn = random.randint(nsyn_min, nsyn_max)
    return tmp_nsyn

In [12]:

# The cparameters dictionary is the connection parameters dictionary which will be 
# used between connected nodes. In this API, this is how arguments/variables are given to
# the connection function (defined earlier and will be used below). 
# Here the function that will use these parameters was defined at the start of the notebook
# and was called distance_connection_handler().

cparameters = {'d_max': 160.0,    # Maximum separation between nodes where connection allowed 
               'nsyn_min': 1,     # If connection exist, minimum number of synapses
               'nsyn_max': 1}     # If connection exist, maximum number of synapses

In [13]:
'''
need to make special axon for interneurons


Rules needed:
1. IN_on distal dendrite to TC_on dendrite, soma
2. IN_on axon to TC_on dendrite, soma
3. Br_on axon to IN_on proximal dendrite
4. Br_on axon to IN_on distal dendrite

1. and 4. need to be near each other



for later: 
5. IN_on distal denrite to IN_off proximal dendrite 
6. IN_on axon to IN_off proximal dendrite, soma
7. IN_off distal denrite to IN_on proximal dendrite
8. IN_off axon to IN_on proximal dendrite, soma
9. Br_on axon to IN_off proximal dendrite
10. Br_off axon to IN_on proximal dendrite
11. Br_on axon to IN_on proximal dendrite
12. Br_off axon to IN_off poximal dendrite 

'''

'\nneed to make special axon for interneurons\n\n\nRules needed:\n1. IN_on distal dendrite to TC_on dendrite, soma\n2. IN_on axon to TC_on dendrite, soma\n3. Br_on axon to IN_on proximal dendrite\n4. Br_on axon to IN_on distal dendrite\n\n1. and 4. need to be near each other\n\n\n\nfor later: \n5. IN_on distal denrite to IN_off proximal dendrite \n6. IN_on axon to IN_off proximal dendrite, soma\n7. IN_off distal denrite to IN_on proximal dendrite\n8. IN_off axon to IN_on proximal dendrite, soma\n9. Br_on axon to IN_off proximal dendrite\n10. Br_off axon to IN_on proximal dendrite\n11. Br_on axon to IN_on proximal dendrite\n12. Br_off axon to IN_off poximal dendrite \n\n'

In [14]:
# Here we will give a connection rule we want to impose on the network. Note that
# we only give the rules (with associated functions) and the network object created earlier. The API
# will then loop and determine the connections that satify the rules.

net.add_edges(
    # Connection rule states A: Everytime the source node (pre-synaptic neuron) is inhibitory
    source={'ei': 'i'}, 
    # Connection rule states B: Everytime the target node (post-synamtic neuron) is
    # inhibitory and of biophysical detail. One can change this of course to another
    # rule that depends on pop_name or node_mode_id only for instance.
    target={'ei': 'e', 'model_type': 'biophysical'},
    # Function that will determine the rules of connectivity. This was defined before
    # and required 5 arguments. However, note that the source node and target
    # node will be automatically set as arguments to the function and hence only
    # need 3 arguments. 
    connection_rule=distance_connection_handler,
    # The remaining arguments needed by distance_connection_handler that we defined above.
    connection_params=cparameters,
    # Till now we only determined if node X is connected to node Y.
    # The parameters of the connections are defined below (edge_parameters)
    # syn_weight defines the strength of the connection. It depends on the 
    # target cell mechanism. Here, this will be interpreted as the peak
    # conductance measure in uS.
    syn_weight=0.0004, 
    # When the target is a biophysical neuron, the location of the connections 
    # (synapses) along the neuron must be defined.
    distance_range=[0.0, 1e+20],
    # Further, when the target is a biophysical neuron, the regions of the 
    # neuron being targeted need to be defined.
    target_sections=['somatic', 'basal'], 
    # Axonal delay between when pre-synaptic node fires and post-synaptic
    # target responds. The unit is milliseconds
    delay=2.0,
    # File that contains the parameters for the synaptic weight which you can
    # explore in the components directory. The naming convention below indicates
    # that the synapse dynamics are GABA for an Inh to Inh node.
    dynamics_params='GABA_InhToExc.json', 
    # params_file only has parameter values, but the mechanism (profile) of
    # synaptic dynamics are defined in NEURON. Here we use exp2syn where you
    # can find more info here:
    # https://www.neuron.yale.edu/neuron/static/docs/help/neuron/neuron/mech.html#Exp2Syn
    model_template='exp2syn'
)

In [15]:

# This will actually build the network and determine which nodes are connected to which. 
# Until now, only the rules were given and stored.
net.build()

In [16]:
# Save the edges and edge_types file.
net.save_edges(edges_file_name='edges.h5', edge_types_file_name='edge_types.csv', output_dir=directory_name)

In [17]:
# Viewing the edge_types file
edge_types_DF = pd.read_csv('network/recurrent_network/edge_types.csv', sep = ' ')
edge_types_DF

,edge_type_id,target_query,source_query,distance_range,dynamics_params,model_template,syn_weight,target_sections,delay
0,100,ei=='e'&model_type=='biophysical',ei=='i',"[0.0, 1e+20]",GABA_InhToExc.json,exp2syn,0.0004,"['somatic', 'basal']",2.0


In [18]:
# Will save the output files in a parallel manner to what we did before.
# Output file names
directory_name = 'network/source_input/'

In [19]:

# Create input nodes dictionary. Will use "virtual" in this example and the dictionary 
# has already been created for you below. In the following you will create the nodes
# and save them as was done for the recurrent network. Note that we are making 25 
# external nodes and we will give them Poission spike trains.

filter_models = {
    'inputFilter': {
        'N': 25, 
        'ei': 'e', 
        'pop_name': 'input_filter', 
        'model_type': 'virtual'
    }
}

In [20]:

# Create a network object
inputNetwork = NetworkBuilder("inputNetwork")

In [21]:

# Add each cell type to the network

# SOLUTION 1:
# for model, params in filter_models.items():
#     inputNetwork.add_nodes(**params)

# SOLUTION 2:
inputNetwork.add_nodes(**filter_models['inputFilter'])

In [22]:
inputNetwork.save_nodes(nodes_file_name='nodes.h5', node_types_file_name='node_types.csv', output_dir=directory_name)

In [23]:

# Print the first 5 external nodes
input_nodes_DF=nodes_table('network/source_input/nodes.h5', 'inputNetwork')
input_nodes_DF[:5]

,node_id,node_type_id
0,0,100
1,1,100
2,2,100
3,3,100
4,4,100


In [24]:
# Print the external node types
input_node_types_DF = pd.read_csv('network/source_input/node_types.csv', sep = ' ')
input_node_types_DF

,node_type_id,model_type,ei,pop_name
0,100,virtual,e,input_filter


In [25]:

# Defining a function that will select source sources to connect to the network.
# In this example, all source cells will connect but this gives you an idea of
# how custom functions can be created to select subsets of input sources nodes.
# Also we here introduce a new concept for connecting nodes. Here the function
# receives all sources and a single target. An algorithm can determine which
# sources connect to the given target. The function will return a list of length
# equal to the number of sources. The elements in the list can be either N_syn
# everytime there is a connection or None when there is no connection.
def select_source_cells(sources, target, N_syn):
    '''
    Note here that "sources" are given (not "source"). So the iterations occur through every target 
    with all sources as potential inputs. Faster than before and better if will have common rules.
    '''

    target_id = target.node_id
    source_ids = [s.node_id for s in sources]

    nsyns_ret = [N_syn]*len(source_ids)
    return nsyns_ret

In [26]:

# Similar to as was done for the recurrent network.  
cparams = {'N_syn': 10}

In [27]:
# Connect the external population to the biophysical exctitaory nodes (Scnn1a).
# This is the same format as before, but note that this time we give an input
# to the connect function called "iterator". Previously, when we did not use this
# and the default "one_to_one" was used. When in the default "one_to_one" case, the 
# connector function iterates through every source and target one by one. When
# the iterator is "all_to_one", the connector function will expect all sources
# to be received together for every target. Other options are "one_to_all". The
# final option "all_to_all" is still not available.
inputNetwork.add_edges(target=net.nodes(pop_name='TC'),
                       iterator='all_to_one',
                       connection_rule=select_source_cells,
                       connection_params=cparams,
                       syn_weight=0.0007, 
                       distance_range=[0.0, 150.0],
                       target_sections=['basal', 'apical'],
                       delay=2.0,
                       dynamics_params='AMPA_ExcToExc.json',
                       model_template='exp2syn')

In [28]:

# If want to connect input to inhibitory
inputNetwork.add_edges(target=net.nodes(pop_name='IN'),
                      iterator='all_to_one',
                      connection_rule=select_source_cells,
                      connection_params=cparams,
                      syn_weight=0.002, 
                      distance_range=[0.0, 1.0e+20],
                      target_sections=['basal', 'somatic'],
                      delay=2.0,
                      dynamics_params='AMPA_ExcToInh.json',
                      model_template='exp2syn')

In [29]:
inputNetwork.build()

In [30]:
inputNetwork.save_edges(edges_file_name='edges.h5', edge_types_file_name='edge_types.csv', output_dir=directory_name)

In [31]:
input_edge_types_DF = pd.read_csv('network/source_input/edge_types.csv', sep = ' ')
input_edge_types_DF

,edge_type_id,target_query,source_query,distance_range,dynamics_params,model_template,syn_weight,target_sections,delay
0,100,pop_name=='TC',*,"[0.0, 150.0]",AMPA_ExcToExc.json,exp2syn,0.0007,"['basal', 'apical']",2.0
1,101,pop_name=='IN',*,"[0.0, 1e+20]",AMPA_ExcToInh.json,exp2syn,0.0020,"['basal', 'somatic']",2.0


In [32]:
input_ids = [n.node_id for n in inputNetwork.nodes()]
from bmtk.utils.io.spike_trains import PoissonSpikesGenerator

# Create a Poisson Spike train for all input nodes that fire at a rate of 0.5Hz. 
# The time units below is in milliseconds
psg = PoissonSpikesGenerator(gids=input_ids, firing_rate=0.5, tstart=0.0, tstop=10000.0) 

# Save the spike trains
psg.to_hdf5(file_name='network/source_input/poission_input_spk_train.h5')

In [33]:
import json
import pprint

with open('config.json') as config:
    config_file = json.load(config)

pprint.pprint(config_file)

{'components': {'biophysical_neuron_models_dir': '$COMPONENT_DIR/biophysical/electrophysiology',
                'mechanisms_dir': '$MECHANISMS_DIR',
                'morphologies_dir': '$COMPONENT_DIR/biophysical/morphology',
                'point_neuron_models_dir': '$COMPONENT_DIR/intfire',
                'synaptic_models_dir': '$COMPONENT_DIR/synaptic_models'},
 'conditions': {'celsius': 34.0, 'v_init': -80},
 'inputs': {'spike_trains': {'input_file': '$INPUT_DIR/poission_input_spk_train.h5',
                             'input_type': 'spikes',
                             'module': 'h5',
                             'node_set': 'inputNetwork'}},
 'manifest': {'$BASE_DIR': '${configdir}',
              '$COMPONENT_DIR': '${configdir}/components',
              '$INPUT_DIR': '$BASE_DIR/network/source_input',
              '$MECHANISMS_DIR': '${configdir}/components/mechanisms',
              '$NETWORK_DIR': '$BASE_DIR/network',
              '$OUTPUT_DIR': '$BASE_DIR/output'},
 'n

In [34]:
from bmtk.simulator import bionet


conf = bionet.Config.from_json('config.json', validate=True)
conf.build_env()

graph = bionet.BioNetwork.from_config(conf)
sim = bionet.BioSimulator.from_config(conf, network=graph)
sim.run()

2020-04-22 18:02:41,004 [INFO] Created log file
2020-04-22 18:02:41,090 [INFO] Building cells.
could not open TC_jy170517_A_idA.asc


NEURON: TC_jy170517_A_idA.asc :file is not open
 near line 0
 ^
        File[0].eof()
      Import3d_Neurolucida3[0].rdfile("TC_jy17051...")
    Import3d_Neurolucida3[0].input("TC_jy17051...")
  Biophys1[0].init("TC_jy17051...")


RuntimeError: hoc error

In [35]:
from neuron import h
# Print the names of all density mechanisms
mt = h.MechanismType(0)
mname  = h.ref('')
for i in list(range(int(mt.count()))):
    mt.select(i)
    mt.selected(mname)
    print(mname[0])

morphology
capacitance
pas
extracellular
fastpas
na_ion
k_ion
hh
ca_ion
SK_E2
TC_HH
TC_iT_Des98
TC_ih_Bud97
TC_Nap_Et2
TC_cad
TC_iA
TC_iL


In [39]:
# Load the spikes
spks = np.loadtxt('output/spikes.csv')

# Print the first 10 spikes
print(spks[1:10, :])

Overwriting the output directory C:\Users\yijan\OneDrive\Desktop\Brain2017_backup_200316\SWDB_2017\DynamicBrain\Modeling\biophysical_notebook/sim_results_001:
Created a log file -- on 2020/04/22 at 09:17:26
Output directory: C:\Users\yijan\OneDrive\Desktop\Brain2017_backup_200316\SWDB_2017\DynamicBrain\Modeling\biophysical_notebook/sim_results_001 -- t_wall: 12.9088068 s
Config file: C:\Users\yijan\OneDrive\Desktop\Brain2017_backup_200316\SWDB_2017\DynamicBrain\Modeling\biophysical_notebook\config.json -- t_wall: 12.9092973 s
Number of ranks: 1 -- t_wall: 12.96547 s
Number of nodes: 2 -- t_wall: 12.9684602 s
Set up node properties -- t_wall: 12.9697952 s
Setting up network... -- t_wall: 12.9760247 s


ValueError: argument not a density mechanism name.

In [40]:
# Basic raster plot
plt.plot(spks[:, 0], spks[:, 1], '.k')
plt.xlim(0, config_file['run']['tstop'])
plt.ylim(-1, 7.5)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron Number')

morphology
capacitance
pas
extracellular
fastpas
na_ion
k_ion
hh
ca_ion
SK_E2
TC_HH
TC_iT_Des98
TC_ih_Bud97
TC_Nap_Et2
TC_cad
TC_iA
TC_iL


In [ ]:
# Load the file for a node and see all the recorded variables
import h5py
h5_output_file = 'output/cell_vars.h5'
f = h5py.File(h5_output_file, 'r')
for key in f:
    if key.lower() == 'mapping':
        continue
    print('Variables recorded:', key)

In [ ]:

# Select the membrane voltage variable and plot it
node = 2
mem_vol = np.array(f['/v/data'])
time = np.arange(0, config_file['run']['tstop'], config_file['run']['dt'])
plt.plot(time, mem_vol[:,node])
plt.xlabel('Time (ms)')
plt.ylabel('Membrane voltage (mV)')